# Paired L9 check after U3

This checks whether the first forget request makes later learning of task 9 more damaging to retained task 0. One arm learns L9 from the E08 checkpoint without U3. The other learns L9 from the saved post-U3 checkpoint. Both use the same task data, seed, and learning settings.

The notebook also repeats L9 from the post-U3 checkpoint. The original full sequence gave a different L9 result from the first replay, so the paired difference is not evidence about U3 until that mismatch is understood. All runs, checks, and the comparison are saved to Drive before the GPU runtime is released.


In [ ]:
from urllib.request import urlretrieve
import runpy

urlretrieve(
    "https://raw.githubusercontent.com/sumitasthana/CARK/main/scripts/colab_setup.py",
    "/content/uncle_colab_setup.py",
)
_ = runpy.run_path("/content/uncle_colab_setup.py", run_name="__main__")


## Check both starting points

The E08 checkpoint is before U3. The second checkpoint is immediately after U3 in the completed sequence. This cell checks their request histories, accuracies, and settings before any GPU training.


In [ ]:
from dataclasses import replace
from pathlib import Path
from uncle.config import Config
from uncle.notebook import inspect_checkpoint, prepare_session

ROOT = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace")
E08 = ROOT / "before_forgetting.pt"
POST_U3 = ROOT / "full_sequence_27step_candidate/checkpoint_seq1_resnet50_seed0_after_request02_forget3.pt"
DATA = Path("/content/data/tiny-imagenet-200")
OUTPUT = ROOT / "paired_L9_probe"

e08 = prepare_session(
    E08, expected_history=[("learn", "3"), ("learn", "0")],
    expected_accuracies={"3": 26.0, "0": 44.6},
    expected_config={"dataset": "tiny_imagenet", "backbone": "resnet50"},
)
post = inspect_checkpoint(
    POST_U3, expected_history=[("learn", "3"), ("learn", "0"), ("forget", "3")],
    expected_accuracies={"3": 12.6, "0": 41.0},
    expected_config={"dataset": "tiny_imagenet", "backbone": "resnet50"},
)
assert e08["forgotten"] == [] and post["forgotten"] == ["3"]
full_config = Config(**post["config"])
assert full_config.requests[:4] == (("learn", "3"), ("learn", "0"),
                                    ("forget", "3"), ("learn", "9"))
assert (full_config.gamma, full_config.forgetting_learning_rate,
        full_config.burn_in, full_config.burn_in_decay, full_config.burn_in_min) == (5e-6, 1e-5, 27, 1.0, 27)
assert e08["config"]["seed"] == full_config.seed
assert e08["config"]["tasks"] == post["config"]["tasks"]
without_u3 = replace(full_config, requests=(("learn", "3"), ("learn", "0"), ("learn", "9")))
after_u3 = replace(full_config, requests=full_config.requests[:4])
print("Without U3:", E08, "then L9")
print("After U3:", POST_U3, "then L9")
print("Output:", OUTPUT)


## Run L9 once from each checkpoint

Each arm has a separate resumable checkpoint. If Colab stops between arms, run these cells again in a fresh GPU session. Completed arms resume without repeating L9.


In [ ]:
from uncle.experiments import run_experiment

arms = {
    "without_u3": (E08, without_u3),
    "after_u3": (POST_U3, after_u3),
}
runs = {}
for name, (source, config) in arms.items():
    print("Running", name, flush=True)
    runs[name] = run_experiment(
        sequence=1, config=config, starting_checkpoint=source,
        root=DATA, download=True, output=OUTPUT / name,
        checkpoint=True, resume=True, verbose=True, progress=True,
    )


## Save the paired result

Positive task 0 loss means L9 reduced task 0 accuracy. The difference between arms is only a candidate effect. The replay audit below checks whether the post-U3 arm reproduces the original L9 result.


In [ ]:
import json
import os

reference_path = ROOT / "full_sequence_27step_candidate/history_seq1_resnet50_seed0.json"
reference = json.loads(reference_path.read_text(encoding="utf-8"))[3]
assert (reference["action"], reference["task"]) == ("learn", "9")
comparison = {
    "status": "complete",
    "question": "Does U3 increase task 0 accuracy loss during L9?",
    "arms": {},
    "original_full_run_l9": {
        "task0_before": reference["before"]["0"],
        "task0_after": reference["after"]["0"],
        "task9_after": reference["after"]["9"],
    },
}
for name, (source, config) in arms.items():
    result = runs[name]
    assert len(result["history"]) == len(config.requests)
    row = result["history"][-1]
    assert (row["action"], row["task"]) == ("learn", "9")
    stem = f"seq1_{config.backbone}_seed{config.seed}"
    origin = json.loads((OUTPUT / name / f"origin_{stem}.json").read_text())
    comparison["arms"][name] = {
        "source_checkpoint": str(source),
        "source_sha256": origin["starting_checkpoint_sha256"],
        "task0_before": row["before"]["0"],
        "task0_after": row["after"]["0"],
        "task0_loss_points": row["before"]["0"] - row["after"]["0"],
        "task9_after": row["after"]["9"],
        "task3_before": row["before"]["3"],
        "task3_after": row["after"]["3"],
        "git_commit": result["environment"]["git_commit"],
    }
comparison["extra_task0_loss_after_u3"] = (
    comparison["arms"]["after_u3"]["task0_loss_points"]
    - comparison["arms"]["without_u3"]["task0_loss_points"]
)
OUTPUT.mkdir(parents=True, exist_ok=True)
report_path = OUTPUT / "paired_L9_comparison.json"
temporary = report_path.with_suffix(".json.writing")
temporary.write_text(json.dumps(comparison, indent=2) + "\n", encoding="utf-8")
os.replace(temporary, report_path)
print(json.dumps(comparison, indent=2))
print("Saved:", report_path)


## Repeat L9 from the post-U3 checkpoint

This is a separate resumable run. It uses the same source checkpoint and settings as the first post-U3 replay. If it matches that replay but both differ from the original sequence, the mismatch is tied to the original run or its environment. If the two replays differ, we need to investigate run-to-run variation.


In [ ]:
repeat_output = OUTPUT / "after_u3_repeat"
repeat = run_experiment(
    sequence=1, config=after_u3, starting_checkpoint=POST_U3,
    root=DATA, download=True, output=repeat_output,
    checkpoint=True, resume=True, verbose=True, progress=True,
)
assert len(repeat["history"]) == len(after_u3.requests)
assert (repeat["history"][-1]["action"], repeat["history"][-1]["task"]) == ("learn", "9")


## Review and save the replay audit

The audit checks source checkpoint identity, request history, settings, task sizes, and runtime details. It compares task 0 accuracy after L9 across the original run and both post-U3 replays. Matching task sizes do not prove the image files are identical.


In [ ]:
import hashlib
import pandas as pd
from uncle import checkpoint as checkpointing

stem = f"seq1_{after_u3.backbone}_seed{after_u3.seed}"
original_folder = ROOT / "full_sequence_27step_candidate"
original_history = json.loads((original_folder / f"history_{stem}.json").read_text(encoding="utf-8"))
original_environment = json.loads((original_folder / f"environment_{stem}.json").read_text(encoding="utf-8"))
source_saved = checkpointing.load(POST_U3)
assert source_saved is not None

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def saved_run(folder):
    return {
        kind: json.loads((folder / f"{kind}_{stem}.json").read_text(encoding="utf-8"))
        for kind in ("origin", "environment", "history", "costs", "summary")
    }

run_files = {name: saved_run(OUTPUT / name) for name in arms}
run_files["after_u3_repeat"] = saved_run(repeat_output)
source_sha = sha256_file(POST_U3)
source_config = json.loads(json.dumps(source_saved["config"]))
checks = {
    "source_matches_original_history_prefix": source_saved["history"] == original_history[:3],
    "source_config_matches_original": source_config == original_environment["config"],
    "replay_source_hashes_match": all(
        run_files[name]["origin"]["starting_checkpoint_sha256"] == source_sha
        for name in ("after_u3", "after_u3_repeat")
    ),
    "replay_settings_match": run_files["after_u3"]["environment"]["config"]
                             == run_files["after_u3_repeat"]["environment"]["config"],
    "replay_task9_sizes_match_original": all(
        run_files[name]["environment"]["task_sizes"]["9"]
        == original_environment["task_sizes"]["9"]
        for name in ("after_u3", "after_u3_repeat")
    ),
}
runtime_keys = ("torch", "cuda", "gpu")
runtime_details = {
    name: {key: env.get(key) for key in runtime_keys}
    for name, env in {
        "original": original_environment,
        **{name: files["environment"] for name, files in run_files.items()},
    }.items()
}
checks["replay_runtime_matches_original"] = all(
    runtime_details[name] == runtime_details["original"]
    for name in ("after_u3", "after_u3_repeat")
)

rows = []
for name, files in run_files.items():
    row = files["history"][-1]
    last_cost = next(cost for cost in files["costs"]["requests"]
                     if cost["index"] == len(files["history"]) - 1)
    assert (row["action"], row["task"]) == ("learn", "9")
    rows.append({
        "arm": name, "task0_before": row["before"]["0"],
        "task0_after": row["after"]["0"],
        "task0_loss_points": row["before"]["0"] - row["after"]["0"],
        "task9_after": row["after"]["9"],
        "L9_seconds": last_cost["seconds"],
        "git_commit": files["environment"]["git_commit"],
    })
original_l9 = original_history[3]
rows.append({
    "arm": "original_full_run",
    "task0_before": original_l9["before"]["0"],
    "task0_after": original_l9["after"]["0"],
    "task0_loss_points": original_l9["before"]["0"] - original_l9["after"]["0"],
    "task9_after": original_l9["after"]["9"],
    "L9_seconds": None,
    "git_commit": original_environment["git_commit"],
})
display(pd.DataFrame(rows))

first = run_files["after_u3"]["history"][-1]["after"]["0"]
second = run_files["after_u3_repeat"]["history"][-1]["after"]["0"]
original = original_l9["after"]["0"]
audit = {
    "status": "complete",
    "source_checkpoint": str(POST_U3),
    "source_sha256": source_sha,
    "checks": checks,
    "runtime_details": runtime_details,
    "task0_after_l9": {
        "original_full_run": original,
        "after_u3_first_replay": first,
        "after_u3_second_replay": second,
    },
    "replay_difference_points": second - first,
    "first_replay_difference_from_original_points": first - original,
    "second_replay_difference_from_original_points": second - original,
    "note": "The paired U3 effect is inconclusive until the original L9 replay mismatch is explained.",
}
audit_path = OUTPUT / "paired_L9_replay_audit.json"
temporary = audit_path.with_suffix(".json.writing")
temporary.write_text(json.dumps(audit, indent=2) + "\n", encoding="utf-8")
os.replace(temporary, audit_path)
print("Checks:", checks)
print("Post-U3 replay difference:", second - first, "points")
print("First replay versus original:", first - original, "points")
print("Second replay versus original:", second - original, "points")
print("Saved:", audit_path)


## Release the GPU after saving

This verifies all three runs and both reports, flushes Drive writes, and disconnects the GPU runtime. If a check fails, the runtime stays connected so the saved files can be inspected.


In [ ]:
from google.colab import drive, runtime

assert json.loads(report_path.read_text(encoding="utf-8")) == comparison
assert json.loads(audit_path.read_text(encoding="utf-8")) == audit
for name in ("without_u3", "after_u3", "after_u3_repeat"):
    folder = OUTPUT / name
    for filename in (f"checkpoint_{stem}.pt", f"origin_{stem}.json",
                     f"environment_{stem}.json", f"history_{stem}.json",
                     f"costs_{stem}.json", f"summary_{stem}.json"):
        saved_path = folder / filename
        assert saved_path.is_file() and saved_path.stat().st_size > 0, saved_path
    saved_history = json.loads((folder / f"history_{stem}.json").read_text(encoding="utf-8"))
    expected_length = 3 if name == "without_u3" else 4
    assert len(saved_history) == expected_length
    assert (saved_history[-1]["action"], saved_history[-1]["task"]) == ("learn", "9")
print("All L9 runs, comparison, and replay audit saved. Flushing Drive writes.", flush=True)
drive.flush_and_unmount(timeout_ms=600000)
print("Drive flushed. Releasing GPU runtime.", flush=True)
runtime.unassign()
